In [1]:
"""
Author: Marco Montemore
Affiliation: Undergraduate Research Assistant & Student
Organization: University of Oregon - DSCI 410L (DSiA)
Date: 04/24/2026
Description: This program is designed to clean, filter, and parse mobile crisis and first responder
             data in Eugene/Springfield, Oregon.
""";

In [2]:
# Importing Necessary Libraries.
import pandas as pd

# Cleaning Visual Crossing Weather Data:

1. Capitalize and space-format column titles accordingly.

2. Ensure all datatypes are correct, in that “DateTime” is a DateTime object, “Precipitation Type” is a string, “Snow”/”Snow Depth” are floats, and “Conditions” is a string.

3. Drop unimportant columns. Can always re-add in the event of analysis change.

4. Merge the cleaned dataset to each of the above datasets on the DateTime-attributed columns.


In [3]:
# First, read in the weather data.
weather_data = pd.read_csv("Weather_Data/weather_data.csv")

# Renaming columsn accordingkly for formatting purposes.
weather_data.rename(columns={
    "datetime": "DateTime",
    "snow": "Snow",
    "preciptype": "Precipitation Type",
    "snowdepth": "Snow Depth",
    "conditions": "Conditions"
}, inplace=True)

# Drop unecessary columns (Snow, Conditions)
weather_data.drop(columns=["Snow", "Conditions"], inplace=True)

# In the preciptype column, if the value is simple ",," or simply nothing, replace with "nopn" to indicate no precipitation type.
weather_data["Precipitation Type"] = weather_data["Precipitation Type"].replace({None: "nopn"})
weather_data["Precipitation Type"] = weather_data["Precipitation Type"].replace({"": "nopn"})

# Ensure all datatypes are correct for analysis.
weather_data["DateTime"] = pd.to_datetime(weather_data["DateTime"])
weather_data["Precipitation Type"] = weather_data["Precipitation Type"].astype(str)
weather_data["Snow Depth"] = weather_data["Snow Depth"].astype(float)

# Save the cleaned data to a new CSV file for future use.
weather_data.to_csv("Weather_Data/weather_CLEAN.csv", index=False)
weather_data.head()

,DateTime,Precipitation Type,Snow Depth
0,2015-01-01,nopn,0.0
1,2015-01-02,snow,0.0
2,2015-01-03,nopn,0.0
3,2015-01-04,rain,0.0
4,2015-01-05,nopn,0.0


# Cleaning Eugene CAD Data

### All Changes to data made:

1. 2015 - 2025 Data Merged.

2. "case_id" changed to string dtype for consistency.

3. "calltime" converted to datetime object for consistency.

4. "month" column in 2025 data dropped to accurately merge data.

5. Columns not useful to study including "yr", "agency", "priority", "service", "case_id", "callsource", "closecode", "secs_to_close", "disp", "arrv", "units_dispd", "units_arrived" in merged data dropped.

6. All rows with primeunit = "NaN" dropped, as an identifier is needed to know the type of responder. Calls where an actual response occurs is what matters for the analysis.

7. Capitalize and remove "_" characters in column titles for proper formatting.

8. Change Nature, Closed As, and Prime Unit columns from object to string data types.

9. Merge weather data to dataframe on datetime-attributed column.

10. Check for duplicates. Remove trailing spaces in the values of the Prime Unit column.

11. Make a binary classifier column for "Is CAHOOTS" based on known designations of CAHOOTS services relayed from classmates and the DSiA team to make later analysis easier.

12. Save and export new dataframe to CSV format.


In [4]:
# Need to merge the 2015 - 2025 csvs together into one massive csv for easier analysis.
# List of years to process. 2026 is shown below but is actually for the 2025 data due to Python's range function excluding the stop value.
years = list(range(2015, 2026))

# List to hold DataFrames
dfs = []
# Loop through each year, read the corresponding CSV, and append it to the list
for year in years:

    # Make the current year's CSV into a DataFrame and append it to the list of DataFrames.
    df = pd.read_csv(
        f"Eugene_CAD_Data/{year}_CAD.csv",
        dtype = {"case_id": "string"},
        parse_dates = ["calltime"],
        low_memory = False)
    
    if "month" in df.columns:
        df = df.drop(columns = ["month"])

    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

In [5]:
# Now cleaning merged dataframe.
# Removing unecessary columns.
less_columns = merged_df.drop(columns = ["yr", "service", "case_id", "callsource", "priority", "closecode", "secs_to_disp", "secs_to_close", "disp", "arrv", "units_dispd", "units_arrived"])
# Remove all rows where "primeunit" or "closed_as" is equal to Null/NA.
cleaned_df = less_columns.dropna(subset = ["primeunit", "closed_as"])
# Rename columns to be capitalized.
cleaned_df = cleaned_df.rename(columns = {"inci_id": "ID", "agency": "Agency", "calltime": "Call Time", "nature": "Nature", "closed_as": "Closed As", "secs_to_arrv": "Seconds to Arrival", "primeunit": "Prime Unit"})

# Change Nature, Closed As, and Prime Unit to string data types.
cleaned_df["Nature"] = cleaned_df["Nature"].astype("string")
cleaned_df["Closed As"] = cleaned_df["Closed As"].astype("string")
cleaned_df["Prime Unit"] = cleaned_df["Prime Unit"].astype("string")

# Count number of nans in each column.
nan_counts = cleaned_df.isna().sum()
# Print the counts.
print(len(cleaned_df))
print(nan_counts)
cleaned_df.head()

1091094
Agency                     0
ID                         0
Call Time                  0
Nature                     0
Closed As                  0
Seconds to Arrival    180608
Prime Unit                 0
dtype: int64


,Agency,ID,Call Time,Nature,Closed As,Seconds to Arrival,Prime Unit
0,EPD,15000001,2015-01-01 00:00:00,PERSON STOP,ASSISTED,0.0,_5E48
1,EPD,15000002,2015-01-01 00:00:44,FIGHT,RESOLVED,0.0,_3F65
2,EPD,15000003,2015-01-01 00:01:05,CHECK WELFARE,ASSISTED,1094.0,_3J79
3,EPD,15000007,2015-01-01 00:03:16,SHOTS FIRED,PATROL CHECK,576.0,_5E48
4,EPD,15000010,2015-01-01 00:03:34,ILLEGAL FIREWORKS,ADVISED,0.0,_5K97


In [6]:
# Merge the cleaned CAD data with the cleaned weather data on dummy columns based on "Call Time" and "DateTime" columns, respectively.
cleaned_df["Call Date"] = cleaned_df["Call Time"].dt.floor("D")
weather_data["Date"] = pd.to_datetime(weather_data["DateTime"]).dt.floor("D")

# Now merge the two DataFrames on the dummy date columns.
# Print datatype of Call Date and DateTime to ensure they are the same for merging.
merged_weather_cad = pd.merge(cleaned_df, weather_data, left_on = "Call Date", right_on = "Date", how = "left")
# Drop dummy columns after merging.
merged_weather_cad = merged_weather_cad.drop(columns = ["Call Date", "DateTime", "Date"])
# Check for duplicate rows in the merged DataFrame.
duplicates = merged_weather_cad.duplicated()
print(f"Number of duplicate rows in the merged DataFrame: {duplicates.sum()}")
merged_weather_cad.head()

Number of duplicate rows in the merged DataFrame: 0


,Agency,ID,Call Time,Nature,Closed As,Seconds to Arrival,Prime Unit,Precipitation Type,Snow Depth
0,EPD,15000001,2015-01-01 00:00:00,PERSON STOP,ASSISTED,0.0,_5E48,nopn,0.0
1,EPD,15000002,2015-01-01 00:00:44,FIGHT,RESOLVED,0.0,_3F65,nopn,0.0
2,EPD,15000003,2015-01-01 00:01:05,CHECK WELFARE,ASSISTED,1094.0,_3J79,nopn,0.0
3,EPD,15000007,2015-01-01 00:03:16,SHOTS FIRED,PATROL CHECK,576.0,_5E48,nopn,0.0
4,EPD,15000010,2015-01-01 00:03:34,ILLEGAL FIREWORKS,ADVISED,0.0,_5K97,nopn,0.0


In [7]:
# Remove trailing spaces in the values of the Prime Unit column.
merged_weather_cad["Prime Unit"] = merged_weather_cad["Prime Unit"].str.strip()
# "Is CAHOOTS" binary classifier designation in the Prime Unit column.
merged_weather_cad["Is CAHOOTS"] = (
    merged_weather_cad["Prime Unit"].str.contains(r'_.J..', na=False) |
    (merged_weather_cad["Prime Unit"] == "_CAHOT") |
    (merged_weather_cad["Agency"] == "CAHE")
).astype(int)

In [8]:
# Save the merged DataFrame to a new CSV file for future analysis.
merged_weather_cad.to_csv("Eugene_CAD_Data/CAD_CLEAN.csv", index = False)
merged_weather_cad.head()

,Agency,ID,Call Time,Nature,Closed As,Seconds to Arrival,Prime Unit,Precipitation Type,Snow Depth,Is CAHOOTS
0,EPD,15000001,2015-01-01 00:00:00,PERSON STOP,ASSISTED,0.0,_5E48,nopn,0.0,0
1,EPD,15000002,2015-01-01 00:00:44,FIGHT,RESOLVED,0.0,_3F65,nopn,0.0,0
2,EPD,15000003,2015-01-01 00:01:05,CHECK WELFARE,ASSISTED,1094.0,_3J79,nopn,0.0,1
3,EPD,15000007,2015-01-01 00:03:16,SHOTS FIRED,PATROL CHECK,576.0,_5E48,nopn,0.0,0
4,EPD,15000010,2015-01-01 00:03:34,ILLEGAL FIREWORKS,ADVISED,0.0,_5K97,nopn,0.0,0


# Cleaning Springfield SPD Data:

### All Changes to Data Made:

1. Change all time-attributed cell values to datetime objects prior to download.

2. 2015 - 2025 data merged.

3. Drop unecessary columns ("Initial Call Type", "Responding Agency", "Priority", "First Dispatched Time", "Clear Time", "Call Creation Mechanism").

4. Rename columns as "Incident Number": "ID", "Final Call Type": "Nature", "Primary Responding Unit": "Prime Unit", "Call Creation Time": "Call Time".

5. New column for Seconds to Arrival by subtracting First Arrival Time from Call Time.

6. Get rid of all rows where Nature/Prime Unit is equal to Null/NA.

7. Replace values to "Unknown" where Prime Unit is equal to the 6-space placeholder element present in many observations.

8. Change Nature/Prime Unit to string data types.

9. Drop rows where prime unit is unkown.

10. Add "Closed As" column to SPD data from new call designation dataset provided by the DSiA team.

11. Merge weather data to dataframe on datetime-attributed column.

12. Filter out rows that are designated as Not EPD or CAHOOTS. Also, check for duplicates.

13. Make a binary classifier column for "Is CAHOOTS" based on known designations of CAHOOTS services relayed from classmates and the DSiA team to make later analysis easier.

14. Sort columns to make them similar to the CAD data.

15. Save and export the new dataframe to CSV format.

In [9]:
# Need to merge the 2015 - 2025 csvs together into one massive csv for easier analysis.
# List of years to process. 2026 is shown below but is actually for the 2025 data due to Python's range function excluding the stop value.
years = list(range(2015, 2026))

# List to hold DataFrames
dfs = []
# Loop through each year, read the corresponding CSV, and append it to the list
for year in years:

    # Make the current year's CSV into a DataFrame and append it to the list of DataFrames.
    df = pd.read_csv(
        f"Springfield_SPD_Data/{year}_SPD.csv",
        low_memory = False)

    dfs.append(df)

SPD_merged = pd.concat(dfs, ignore_index=True)
SPD_merged.head()

,Incident Number,Initial Call Type,Final Call Type,Responding Agency,Primary Responding Unit,Call Creation Time,First Dispatched Time,First Arrival Time,Clear Time,Priority,Call Creation Mechanism
0,15000074,ALMAUD,AUDIBLE ALARM,SPD,1S18,1/1/15 1:16,1/1/15 1:18,1/1/15 1:21,1/1/15 1:32,3,PHONE
1,15000083,TRFSTP,DWS,SPD,3D2,1/1/15 1:22,1/1/15 1:22,1/1/15 1:22,1/1/15 1:34,4,SELF
2,15000167,MVAUNK,MOTOR VEH ACC UNKNOWN INJ,SPD,,1/1/15 3:15,NaN,NaN,NaN,3,E911
3,15000216,TRFSTP,DWS,SPD,1S22,1/1/15 5:27,1/1/15 5:27,1/1/15 5:27,1/1/15 5:38,4,SELF
4,15000222,SUSPVE,SUSPICIOUS VEHICLE,SPD,1S11,1/1/15 5:52,1/1/15 5:53,1/1/15 5:56,1/1/15 6:04,3,PHONE


In [10]:
# This cell takes about 30 seconds to run.

# Drop unecessary columns.
SPD_less_columns = SPD_merged.drop(columns = ["Initial Call Type", "Priority", "Responding Agency", "First Dispatched Time", "Clear Time", "Call Creation Mechanism"])
# Renaming columns to be capitalized and more concise with previous data.
SPD_cleaned = SPD_less_columns.rename(columns = {"Incident Number": "ID", "Final Call Type": "Nature", "Primary Responding Unit": "Prime Unit", "Call Creation Time": "Call Time"})
# New column for Seconds to Arrival by subtracting First Arrival Time from Call Time.
SPD_cleaned["Call Time"] = pd.to_datetime(SPD_cleaned["Call Time"])
SPD_cleaned["First Arrival Time"] = pd.to_datetime(SPD_cleaned["First Arrival Time"])
SPD_cleaned["Seconds to Arrival"] = (SPD_cleaned["First Arrival Time"] - SPD_cleaned["Call Time"]).dt.total_seconds()
# Drop First Arrival Time column since we now have Seconds to Arrival.
SPD_cleaned = SPD_cleaned.drop(columns = ["First Arrival Time"])

SPD_cleaned.head()

/var/folders/xp/k52bq2g94b94j9qspcl9kvk80000gn/T/ipykernel_23173/3787032645.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  SPD_cleaned["Call Time"] = pd.to_datetime(SPD_cleaned["Call Time"])
/var/folders/xp/k52bq2g94b94j9qspcl9kvk80000gn/T/ipykernel_23173/3787032645.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  SPD_cleaned["First Arrival Time"] = pd.to_datetime(SPD_cleaned["First Arrival Time"])


,ID,Nature,Prime Unit,Call Time,Seconds to Arrival
0,15000074,AUDIBLE ALARM,1S18,2015-01-01 01:16:00,300.0
1,15000083,DWS,3D2,2015-01-01 01:22:00,0.0
2,15000167,MOTOR VEH ACC UNKNOWN INJ,,2015-01-01 03:15:00,NaN
3,15000216,DWS,1S22,2015-01-01 05:27:00,0.0
4,15000222,SUSPICIOUS VEHICLE,1S11,2015-01-01 05:52:00,240.0


In [11]:
# Count values of nans in each column
SPD_nan_counts = SPD_cleaned.isna().sum()
# Get rid of all rows where Nature/Prime Unit is equal to Null/NA.
SPD_cleaned = SPD_cleaned.dropna(subset = ["Nature", "Prime Unit"])
# Check for duplicate rows in the cleaned DataFrame.
SPD_duplicates = SPD_cleaned.duplicated()
print(f"Number of duplicate rows in the cleaned DataFrame: {SPD_duplicates.sum()}")
# Replace all "      " characters in Prime Unit with "Unknown".
SPD_cleaned["Prime Unit"] = SPD_cleaned["Prime Unit"].replace("      ", "Unknown")
# Drop rows where Prime Unit is equal to "Unknown" since we won't be able to classify them as CAHOOTS or not.
SPD_cleaned = SPD_cleaned[SPD_cleaned["Prime Unit"] != "Unknown"]

# Change Nature/Prime Unit to string data types.
SPD_cleaned["Nature"] = SPD_cleaned["Nature"].astype("string")
SPD_cleaned["Prime Unit"] = SPD_cleaned["Prime Unit"].astype("string")
SPD_cleaned.head()

Number of duplicate rows in the cleaned DataFrame: 0


,ID,Nature,Prime Unit,Call Time,Seconds to Arrival
0,15000074,AUDIBLE ALARM,1S18,2015-01-01 01:16:00,300.0
1,15000083,DWS,3D2,2015-01-01 01:22:00,0.0
3,15000216,DWS,1S22,2015-01-01 05:27:00,0.0
4,15000222,SUSPICIOUS VEHICLE,1S11,2015-01-01 05:52:00,240.0
5,15000234,AUDIBLE ALARM,2S11,2015-01-01 07:49:00,480.0


In [12]:
# Merge weather data to SPD data on dummy columns based on "Call Time" and "DateTime" columns, respectively.
SPD_cleaned["Call Date"] = SPD_cleaned["Call Time"].dt.floor("D")
weather_data["Date"] = pd.to_datetime(weather_data["DateTime"]).dt.floor("D")

# Now merge the two DataFrames on the dummy date columns.
SPD_merged_weather = pd.merge(SPD_cleaned, weather_data, left_on = "Call Date", right_on = "Date", how = "left")

In [13]:
# Drop dummy columns after merging.
SPD_merged_weather = SPD_merged_weather.drop(columns = ["Call Date", "DateTime", "Date"])

In [14]:
# Filtering SPD data to only include Prime Units that are NOT the following: "3E+34", "2W71", "2Y19", 3V10, 3X33, 
# CNTR, UO327, Z23, Z24, Z25, Z26, Z30, Z4, Z67, Z9, Anthing with an 'E' (i.e. 2E14). After stripping trailing spaces, 
# these are the Prime Units that need to be filtered out.

SPD_merged_weather["Prime Unit"] = SPD_merged_weather["Prime Unit"].str.strip()
units_to_exclude = ["3E+34", "2W71", "2Y19", "3V10", "3X33", "CNTR", "UO327", "Z23", "Z24", "Z25", "Z26", "Z30", "Z4", "Z67", "Z9"]
SPD_filtered = SPD_merged_weather[
    ~(
        SPD_merged_weather["Prime Unit"].isin(units_to_exclude) |
        SPD_merged_weather["Prime Unit"].str.contains(r"E", na=False)
    )
]

In [15]:
# Binary classifier for whether the Prime Unit is CAHOOTS or not with designation 3J81.
SPD_w_Binary = SPD_filtered.copy()
SPD_w_Binary["Is CAHOOTS"] = (SPD_w_Binary["Prime Unit"] == "3J81").astype(int)
SPD_w_Binary

,ID,Nature,Prime Unit,Call Time,Seconds to Arrival,Precipitation Type,Snow Depth,Is CAHOOTS
0,15000074,AUDIBLE ALARM,1S18,2015-01-01 01:16:00,300.0,nopn,0.0,0
1,15000083,DWS,3D2,2015-01-01 01:22:00,0.0,nopn,0.0,0
2,15000216,DWS,1S22,2015-01-01 05:27:00,0.0,nopn,0.0,0
3,15000222,SUSPICIOUS VEHICLE,1S11,2015-01-01 05:52:00,240.0,nopn,0.0,0
4,15000234,AUDIBLE ALARM,2S11,2015-01-01 07:49:00,480.0,nopn,0.0,0
...,...,...,...,...,...,...,...,...
488063,25326475,DISABLED VEHICLE,1S23,2025-12-30 21:15:00,0.0,nopn,0.0,0
488064,25326480,TRAFFIC STOP,1S23,2025-12-30 21:25:00,0.0,nopn,0.0,0
488065,25326519,CHECK WELFARE,3J81,2025-12-30 22:24:00,420.0,nopn,0.0,1
488066,25326528,AUDIBLE ALARM,1S21,2025-12-30 22:34:00,600.0,nopn,0.0,0


In [16]:
# To add the closed_as column, merging the key.csv to each year_key.csv on "Incident Number" is necessary.
# First, a for loop will combine all of the year_key.csvs into one DataFrame.
# Then, the key.csv will be read in and merged with the combined year_key DataFrame on "Incident Number" to add the closed_as column to the SPD data.

years = list(range(2015, 2026))

# List to hold DataFrames
dfs = []
# Loop through each year, read the corresponding CSV, and append it to the list
for year in years:

    # Make the current year's CSV into a DataFrame and append it to the list of DataFrames.
    df = pd.read_csv(
        f"Springfield_SPD_Data/Close_Codes/{year}_key.csv",
        dtype = {"Incident Number": "int"},
        low_memory = False)
    
    dfs.append(df)

# Merge the year_key dataframes.
merged = pd.concat(dfs, ignore_index=True)
# Strip the spaces characters off of the front and back of Close Code values
merged["Close Code"] = merged["Close Code"].str.strip()

# Read in the key.csv file.
key_df = pd.read_csv("Springfield_SPD_Data/Close_Codes/key.csv", dtype = {"Incident Number": "int"}, low_memory = False)

# Merge the merged year_key DataFrame with the key DataFrame on "Incident Number" to add the closed_as column to the SPD data.
key_merged = pd.merge(merged, key_df, on = "Close Code", how = "left")
# Replace NaN values with "Unknown" in the "Definition" column.
key_merged["Definition"] = key_merged["Definition"].fillna("Unknown")
# Change the Definition column to be called "Closed As" for consistency with the CAD data.
key_merged = key_merged.rename(columns = {"Definition": "Closed As", "Incident Number": "ID"})

# Now merge this key_merged DataFrame with the SPD_w_Binary DataFrame on "ID" to add the Closed As column to the SPD data.
final_SPD = pd.merge(SPD_w_Binary, key_merged, on = "ID", how = "left")

# Drop unecessary column Close Code.
final_SPD = final_SPD.drop(columns = ["Close Code"])

# Order the columns accordingly for easier analysis with the CAD data.
final_SPD = final_SPD[["ID", "Call Time", "Nature", "Closed As", "Seconds to Arrival", "Prime Unit", "Precipitation Type", "Snow Depth", "Is CAHOOTS"]]

In [17]:
# Save the cleaned and filtered SPD data to a new CSV file for future analysis.
final_SPD.to_csv("Springfield_SPD_Data/SPD_CLEAN.csv", index = False)
final_SPD.head()

,ID,Call Time,Nature,Closed As,Seconds to Arrival,Prime Unit,Precipitation Type,Snow Depth,Is CAHOOTS
0,15000074,2015-01-01 01:16:00,AUDIBLE ALARM,Building Check Secure,300.0,1S18,nopn,0.0,0
1,15000083,2015-01-01 01:22:00,DWS,Uniform Traffic Citation Issued,0.0,3D2,nopn,0.0,0
2,15000216,2015-01-01 05:27:00,DWS,Uniform Traffic Citation Issued,0.0,1S22,nopn,0.0,0
3,15000222,2015-01-01 05:52:00,SUSPICIOUS VEHICLE,Information Only,240.0,1S11,nopn,0.0,0
4,15000234,2015-01-01 07:49:00,AUDIBLE ALARM,Building Check Secure,480.0,2S11,nopn,0.0,0


# Cleaning MCSLC Data:

### All Changes to Data Made:

1. Removing all unecessary columns ("End Point of Dispatch", "Dispatch Date & Time", "Arrival on Scene Date & Time", "Engagement with Client Date & Time", "MCIT Departure Date & Time", "Minutes: Arrival ? Departure").

2. Calculating a new column for request -> engagement in seconds for accurate comparison to previous data.

3. Now dropping the original minutes columns ("Minutes: Request ? Dispatch", "Minutes: Dispatch ? Arrival", "Minutes: Arrival ? Engagement").

4. Renaming of columns to reflect similar column names in other agencies' data for consisitency. "Dispatch Request Date & Time": "Call Time", "Reason for Dispatch #1": "Nature", "Reason for Dispatch #2": "Nature #2", "Reason for Dispatch #3": "Nature #3", "Reason for Dispatch #4": "Nature #4", "Reason for Dispatch #5": "Nature #5", "Disposition": "Closed As", "Seconds: Request ? Engagement": "Seconds to Arrival".

5. Filter out all rows where "Seconds to Arrival" or "Nature" is NaN.

6. Changing datatypes accordingly for all remaining columns except ID which is already an integer and Seconds to Arrival which is already a float. String for all excrpt Call Time which is datetime.

7. Drop all rows where City is not Eugene or Springfield... AKA "Other". Also, check for duplicates.

8. Merge weather data to dataframe on datetime-attributed column.

9. Sort columns to make them similar to the CAD data.

10. Save and export the new dataframe to CSV format.

In [18]:
raw_MCSLC = pd.read_csv("MCSLC_Data/MCSLC.csv")
raw_MCSLC.head()

,ID,End Point of Dispatch,City,Dispatch Request Date & Time,Dispatch Date & Time,Arrival on Scene Date & Time,Engagement with Client Date & Time,MCIT Departure Date & Time,Reason for Dispatch #1,Reason for Dispatch #2,Reason for Dispatch #3,Reason for Dispatch #4,Reason for Dispatch #5,Disposition,Minutes: Request ? Dispatch,Minutes: Dispatch ? Arrival,Minutes: Arrival ? Engagement,Minutes: Arrival ? Departure
0,12485,Engaged client,Eugene,8/18/2024 14:12,8/18/2024 14:14,8/18/2024 14:33,8/18/2024 14:34,8/18/2024 15:48,Substance use,NaN,NaN,NaN,NaN,Remained in community,2.0,19.0,1.0,75.0
1,12431,Unable to locate client,Eugene,8/18/2024 15:20,8/18/2024 19:19,8/18/2024 19:31,8/18/2024 0:00,8/18/2024 19:39,Harm/risk of harm to self/others/property,Substance use,NaN,NaN,NaN,Remained in community,NaN,12.0,NaN,8.0
2,12439,Unable to locate client,Eugene,8/18/2024 15:28,8/18/2024 0:00,8/18/2024 0:00,8/18/2024 0:00,8/18/2024 0:00,Substance use,Other,NaN,NaN,NaN,Remained in community,NaN,NaN,NaN,NaN
3,12517,Unable to locate client,Springfield,8/18/2024 15:29,8/18/2024 15:30,8/18/2024 0:00,8/18/2024 15:45,8/18/2024 0:00,Suicidality or suicide attempt,NaN,NaN,NaN,NaN,Remained in community,1.0,NaN,NaN,NaN
4,12487,Engaged client,Springfield,8/18/2024 15:35,8/18/2024 15:46,8/18/2024 16:21,8/18/2024 16:25,8/18/2024 18:56,Suicidality or suicide attempt,NaN,NaN,NaN,NaN,Remained in community,11.0,35.0,4.0,155.0


In [19]:
# First, dropping unecessary columns.
less_MCSLC = raw_MCSLC.drop(columns = ["End Point of Dispatch", "Dispatch Date & Time", "Arrival on Scene Date & Time", "Engagement with Client Date & Time", 
                                       "MCIT Departure Date & Time", "Minutes: Arrival ? Departure"])
less_MCSLC.head(50)                                      
# New column column for minutes between Request and engagement using sum of Minutes: Request ? Dispatxh and Minutes: Dispatch ? Arrival and Minutes: Arrival ? Engagement. 
# If Arrival ? Engagement is NA, then just use sum of Minutes: Request ? Dispatxh and Minutes: Dispatch ? Arrival.
def calculate_seconds(row):
    if pd.isna(row["Minutes: Arrival ? Engagement"]):
        return (row["Minutes: Request ? Dispatch"] + row["Minutes: Dispatch ? Arrival"]) * 60
    else:
        return (row["Minutes: Request ? Dispatch"] + row["Minutes: Dispatch ? Arrival"] + row["Minutes: Arrival ? Engagement"]) * 60

# New column creation using function call.
less_MCSLC["Seconds: Request ? Engagement"] = less_MCSLC.apply(calculate_seconds, axis=1)
# Now dropping the original minutes columns.
cleaned_MCSLC = less_MCSLC.drop(columns = ["Minutes: Request ? Dispatch", "Minutes: Dispatch ? Arrival", "Minutes: Arrival ? Engagement"])
cleaned_MCSLC.head()

,ID,City,Dispatch Request Date & Time,Reason for Dispatch #1,Reason for Dispatch #2,Reason for Dispatch #3,Reason for Dispatch #4,Reason for Dispatch #5,Disposition,Seconds: Request ? Engagement
0,12485,Eugene,8/18/2024 14:12,Substance use,NaN,NaN,NaN,NaN,Remained in community,1320.0
1,12431,Eugene,8/18/2024 15:20,Harm/risk of harm to self/others/property,Substance use,NaN,NaN,NaN,Remained in community,NaN
2,12439,Eugene,8/18/2024 15:28,Substance use,Other,NaN,NaN,NaN,Remained in community,NaN
3,12517,Springfield,8/18/2024 15:29,Suicidality or suicide attempt,NaN,NaN,NaN,NaN,Remained in community,NaN
4,12487,Springfield,8/18/2024 15:35,Suicidality or suicide attempt,NaN,NaN,NaN,NaN,Remained in community,3000.0


In [20]:
# Renaming columns to be more concise and consistent with the cleaned CAD data.
cleaned_MCSLC = cleaned_MCSLC.rename(columns = {"Dispatch Request Date & Time": "Call Time", "Reason for Dispatch #1": "Nature", "Reason for Dispatch #2": "Nature #2", 
                                                "Reason for Dispatch #3": "Nature #3", "Reason for Dispatch #4": "Nature #4", "Reason for Dispatch #5": "Nature #5", 
                                                "Disposition": "Closed As", "Seconds: Request ? Engagement": "Seconds to Arrival"})

In [21]:
# Filter out all rows where Seconds to Arrival is NaN.
final_MCSLC = cleaned_MCSLC.dropna(subset = ["Seconds to Arrival"])
# Filter out all rows where Nature is NaN.
final_MCSLC = final_MCSLC.dropna(subset = ["Nature"])

# Changing datatypes accordingly for all remaining columns except ID which is already an integer and Seconds to Arrival which is already a float. String for all excrpt Call Time which is datetime.
final_MCSLC["City"] = final_MCSLC["City"].astype("string")
final_MCSLC["Call Time"] = pd.to_datetime(final_MCSLC["Call Time"])
final_MCSLC["Nature"] = final_MCSLC["Nature"].astype("string")

# Drop Natures #2 - #5 since they are mostly NaN and we can just use the primary Nature column for analysis.
final_MCSLC = final_MCSLC.drop(columns = ["Nature #2", "Nature #3", "Nature #4", "Nature #5"])

# Drop all rows where City is not Eugene or Springfield... AKA "Other".
final_MCSLC = final_MCSLC[final_MCSLC["City"].isin(["Eugene", "Springfield"])]

In [22]:
# Merge the final MCSLC data with the weather date on the datetime column. Making dummy columns to get an effective merge.
final_MCSLC["Call Date"] = final_MCSLC["Call Time"].dt.floor("D")
# Before merging, filter the weather data to stop at December 17, 2025, as that's the last
# date in the MCSLC data.

weather_data = weather_data[weather_data["Date"] <= "2025-12-17"]

final_MCSLC = pd.merge(final_MCSLC, weather_data, left_on = "Call Date", right_on = "Date", how = "left")

In [23]:
# Drop dummy columns used for merge.
final_MCSLC = final_MCSLC.drop(columns = ["Call Date", "DateTime", "Date"])
# Sort columns to be in the same order as the cleaned CAD data.
final_MCSLC = final_MCSLC[["City", "ID", "Call Time", "Nature", "Closed As", "Seconds to Arrival", "Precipitation Type", "Snow Depth"]]
# Check for duplicates in the final MCSLC data.
duplicates = final_MCSLC.duplicated()
print(f"Number of duplicate rows in the final MCSLC DataFrame: {duplicates.sum()}")

Number of duplicate rows in the final MCSLC DataFrame: 0


In [24]:
# Save the cleaned MCSLC data to a new CSV file for future use.
final_MCSLC.to_csv("MCSLC_Data/MCSLC_CLEAN.csv", index=False)
final_MCSLC.head()

,City,ID,Call Time,Nature,Closed As,Seconds to Arrival,Precipitation Type,Snow Depth
0,Eugene,12485,2024-08-18 14:12:00,Substance use,Remained in community,1320.0,nopn,0.0
1,Springfield,12487,2024-08-18 15:35:00,Suicidality or suicide attempt,Remained in community,3000.0,nopn,0.0
2,Eugene,12290,2024-08-18 19:43:00,Needing social/mental health services,Remained in community,2220.0,nopn,0.0
3,Eugene,12354,2024-08-19 16:29:00,Harm/risk of harm to self/others/property,Remained in community,1620.0,nopn,0.0
4,Eugene,12435,2024-08-19 16:29:00,Other,Remained in community,120.0,nopn,0.0
